# Predictive model -- standing up the LSTM (`TODO.md`: "Predictive model")

`ecolens.forecasting` already has a complete `DemandLSTM` implementation --
model (`models/lstm.py`), feature windowing (`features.py`), training loop
with MLflow logging (`training/train.py`), conformal calibration
(`evaluation/conformal.py`), point-forecast metrics (`evaluation/metrics.py`).
None of it has been proven against real data yet, though.

Its designed data source is the `ml_features_demand_v1` dbt mart -- but that
mart isn't materialized on the real NeonDB warehouse yet: checked directly
against NeonDB before writing this notebook, and `ml`/`analytics`/
`intermediate`/`staging` are all empty schemas there, only `raw.*` is
populated (see `TODO.md`'s PreProcessing section -- the Mongo/DuckDB ->
Postgres syncer this ultimately depends on doesn't exist yet either).

So this notebook builds a *local* stand-in for that mart straight from
`data/historical/ecolens_historical.duckdb` (the same raw AEMO/BoM/holiday
tables `data-pipeline.ipynb` already prototyped `feature_table_30min`
from), replicating `int_energy_unified_30min` -> `int_energy_filled_30min`
-> `int_energy_with_weather` -> `ml_features_demand_v1`'s logic closely
enough to match `forecasting/features.py`'s `FEATURE_COLUMNS` contract
exactly -- then runs the *real* package code (`build_windowed_dataset`,
`train_model`, `evaluate_model`, conformal calibration) against it,
end to end.

Not a byte-for-byte replica of the dbt mart -- the gap-fill below is a
pandas group `ffill` (+ region-mean / global-mean / zero fallback), not
`int_energy_filled_30min`'s SQL "island" trick, and `is_weekend`/
`rain_since_9am_mm`/lag-column generation aren't reproduced since the LSTM
doesn't consume them (see `features.py`'s own exclusion comment). Exact
replication belongs in dbt once the `raw.*` syncer lands and the mart can
be built for real. Good enough here to prove the LSTM pipeline itself is
real, working code -- not scaffolding.

In [ ]:
import logging
import os
from contextlib import contextmanager

import duckdb
import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("EcoLens.PredictiveModel")

# Unlike data-pipeline.ipynb (which never touches Settings/.env), this
# notebook calls ecolens.config.get_settings() further down for the
# model hyperparameters and MLflow tracking URI -- pydantic-settings
# resolves `env_file=".env"` relative to the process CWD, so we chdir to
# the service root once, up front, rather than relying on wherever the
# notebook happens to be launched from.
SERVICE_ROOT = "/Users/macbook/Project/personal/EcoLens/services/data-pipeline"
os.chdir(SERVICE_ROOT)

DEFAULT_DB_PATH = f"{SERVICE_ROOT}/data/historical/ecolens_historical.duckdb"
DB_PATH = os.getenv("DUCKDB_PATH", DEFAULT_DB_PATH)


@contextmanager
def duckdb_connection(path: str, read_only: bool = True):
    """Same helper as data-pipeline.ipynb's -- connect, log, guarantee
    close-once, so this notebook doesn't hand-roll its own copy.
    """
    resolved_path = os.path.expanduser(path)
    if read_only and not os.path.exists(resolved_path):
        logger.error("DuckDB database file not found at path: %s", resolved_path)
        raise FileNotFoundError(f"Database file not found: {resolved_path}")

    con = None
    try:
        con = duckdb.connect(database=resolved_path, read_only=read_only)
        logger.info("Successfully connected (read_only=%s) to %s", read_only, resolved_path)
        yield con
    except duckdb.Error as e:
        logger.exception("DuckDB internal error while connecting to %s", resolved_path)
        raise ConnectionError(f"Failed to connect to DuckDB database: {e}") from e
    finally:
        if con is not None:
            con.close()
            logger.info("DuckDB connection closed cleanly.")


## 1. Build a local `ml_features_demand_v1`-shaped table

Same source-priority logic as `int_energy_unified_30min` (AEMO first,
OpenElectricity fallback -- WEM directly, NEM only for the network-level
fuel-tech mix broadcast onto its 5 sub-regions), trimmed to just the
columns `forecasting/features.py`'s `FEATURE_COLUMNS` needs (plus the few
fuel-tech columns `renewable_generation_mw` sums).

In [ ]:
from datetime import datetime, timezone

START = "2025-08-01 00:00:00"
END = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
REGIONS = ["NSW1", "QLD1", "VIC1", "SA1", "TAS1", "WEM"]

MARKET_COLS = ["demand_mw", "price_mwh", "net_import_mw"]
FUELTECH_COLS = [
    "hydro_mw", "wind_mw", "solar_utility_mw", "solar_rooftop_mw", "biomass_mw",
    "total_generation_mw", "renewable_proportion", "emissions_intensity_kgco2e_per_mwh",
]
WEATHER_COLS = [
    "temp_c", "apparent_temp_c", "dew_point_c", "humidity_pct",
    "wind_speed_kmh", "wind_direction_deg", "wind_gust_kmh",
    "pressure_hpa", "cloud_cover_pct",
]


def _avg_cols(cols: list[str]) -> str:
    return ",\n        ".join(f"AVG({c}) AS {c}" for c in cols)


def _coalesce_market(cols: list[str]) -> str:
    # AEMO first; OpenElectricity only fills a gap for WEM (direct region
    # match) -- it reports NEM at the whole-network level only, so it
    # can't substitute for a missing per-region NEM market row.
    return ",\n    ".join(f"COALESCE(m.{c}, oe_wem.{c}) AS {c}" for c in cols)


def _coalesce_fueltech(cols: list[str]) -> str:
    # AEMO first (WEM's own row, or NEM's broadcast network-level row);
    # OpenElectricity fallback works for both since fuel mix is
    # inherently network-wide on both sides.
    return ",\n    ".join(
        f"COALESCE(CASE WHEN s.region = \'WEM\' THEN w.{c} ELSE f.{c} END, oe.{c}) AS {c}"
        for c in cols
    )


query = f"""
WITH regions AS (
    SELECT * FROM (VALUES {", ".join(f"(\'{r}\')" for r in REGIONS)}) AS t(region)
),
spine AS (
    SELECT r.region, ts_slot
    FROM regions r
    CROSS JOIN (
        SELECT UNNEST(generate_series(
            CAST(? AS TIMESTAMP), CAST(? AS TIMESTAMP) - INTERVAL 30 MINUTE, INTERVAL 30 MINUTE
        )) AS ts_slot
    ) t
),
nem_market_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(MARKET_COLS)}
    FROM aemo_nem_dispatch
    WHERE region != \'NEM\' AND ts >= ? AND ts < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, ts)
),
nem_fueltech_30min AS (
    SELECT time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(FUELTECH_COLS)}
    FROM aemo_nem_dispatch
    WHERE region = \'NEM\' AND ts >= ? AND ts < ?
    GROUP BY time_bucket(INTERVAL 30 MINUTE, ts)
),
wem_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, ts) AS ts_slot,
        {_avg_cols(MARKET_COLS)},
        {_avg_cols(FUELTECH_COLS)}
    FROM aemo_wem_dispatch
    WHERE ts >= ? AND ts < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, ts)
),
oe_30min AS (
    SELECT region, time_bucket(INTERVAL 30 MINUTE, CAST(ts AS TIMESTAMPTZ)) AS ts_slot,
        {_avg_cols(MARKET_COLS)},
        {_avg_cols(FUELTECH_COLS)}
    FROM openelectricity_responses
    WHERE CAST(ts AS TIMESTAMPTZ) >= ? AND CAST(ts AS TIMESTAMPTZ) < ?
    GROUP BY region, time_bucket(INTERVAL 30 MINUTE, CAST(ts AS TIMESTAMPTZ))
),
weather_30min AS (
    SELECT region, ts AS ts_slot,
        {", ".join(f"ANY_VALUE({c}) AS {c}" for c in WEATHER_COLS)}
    FROM bom_observations
    WHERE ts >= ? AND ts < ?
    GROUP BY region, ts
),
holiday_flags AS (
    SELECT region, CAST(date AS DATE) AS holiday_date, TRUE AS is_public_holiday
    FROM aemo_holidays
)
SELECT
    s.region,
    s.ts_slot AS ts,
    {_coalesce_market(MARKET_COLS)},
    {_coalesce_fueltech(FUELTECH_COLS)},
    wx.temp_c, wx.apparent_temp_c, wx.dew_point_c, wx.humidity_pct,
    wx.wind_speed_kmh, wx.wind_direction_deg, wx.wind_gust_kmh,
    wx.pressure_hpa, wx.cloud_cover_pct,
    COALESCE(h.is_public_holiday, FALSE) AS is_public_holiday
FROM spine s
LEFT JOIN nem_market_30min m ON m.region = s.region AND m.ts_slot = s.ts_slot
LEFT JOIN nem_fueltech_30min f ON f.ts_slot = s.ts_slot AND s.region != \'WEM\'
LEFT JOIN wem_30min w ON w.region = s.region AND w.ts_slot = s.ts_slot
LEFT JOIN oe_30min oe ON oe.region = \'NEM\' AND oe.ts_slot = s.ts_slot AND s.region != \'WEM\'
LEFT JOIN oe_30min oe_wem ON oe_wem.region = \'WEM\' AND oe_wem.ts_slot = s.ts_slot AND s.region = \'WEM\'
LEFT JOIN weather_30min wx ON wx.region = s.region AND wx.ts_slot = s.ts_slot
LEFT JOIN holiday_flags h ON h.region = s.region AND h.holiday_date = CAST(s.ts_slot AS DATE)
ORDER BY s.region, s.ts_slot;
"""
# 6 (start, end) pairs above: spine, nem_market, nem_fueltech, wem, oe, weather.
query_params = [START, END] * 6

with duckdb_connection(DB_PATH) as con:
    raw_30min = con.execute(query, query_params).df()

print(f"shape: {raw_30min.shape}")
print("\nrows per region:")
print(raw_30min["region"].value_counts())
print("\nnull % per column (before gap-fill):")
print(raw_30min.isna().mean().round(3))
raw_30min.head()


## 2. Gap-fill, then derive the LSTM's exact feature contract

Forward-fill per region (LOCF, `int_energy_filled_30min`'s intent, done
here as a plain group `ffill`), falling back to that region's own mean,
then the all-region mean, then a hard `0` -- so nothing downstream ever
sees a NaN, same fallback chain as the dbt model, just in pandas.

Then derive `renewable_generation_mw` (mart's own formula), `is_holiday`,
and the cyclical hour/day-of-week/month encodings in each region's local
timezone (`region_timezone_case()`'s mapping, matched here via
`zoneinfo`) -- exactly the columns `FEATURE_COLUMNS` expects.

In [ ]:
FILL_COLS = MARKET_COLS + FUELTECH_COLS + WEATHER_COLS

df = raw_30min.sort_values(["region", "ts"]).reset_index(drop=True)

# A couple of weather columns (wind_direction_deg, cloud_cover_pct) come
# back from duckdb as pandas nullable Int64 rather than float64 (their
# raw values happen to be whole numbers) -- fillna-ing an Int64 column
# from a float64 region-mean/global-mean Series raises a "cannot safely
# cast" TypeError, so normalize every fill column to float64 up front.
df[FILL_COLS] = df[FILL_COLS].astype("float64")

df[FILL_COLS] = df.groupby("region")[FILL_COLS].ffill()
region_means = df.groupby("region")[FILL_COLS].transform("mean")
df[FILL_COLS] = df[FILL_COLS].fillna(region_means)
df[FILL_COLS] = df[FILL_COLS].fillna(df[FILL_COLS].mean())
df[FILL_COLS] = df[FILL_COLS].fillna(0.0)

assert df[FILL_COLS].isna().sum().sum() == 0, "gap-fill left NaNs behind"

df["renewable_generation_mw"] = (
    df["hydro_mw"] + df["wind_mw"] + df["solar_utility_mw"]
    + df["solar_rooftop_mw"] + df["biomass_mw"]
)
df["is_holiday"] = df["is_public_holiday"].astype(int)

print("null % per column (after gap-fill):")
print(df[FILL_COLS].isna().mean().round(3))


In [ ]:
from zoneinfo import ZoneInfo

# Same mapping as macros/time_helpers.sql's region_timezone_case().
REGION_TZ = {
    "NSW1": "Australia/Sydney",
    "QLD1": "Australia/Brisbane",
    "VIC1": "Australia/Melbourne",
    "SA1": "Australia/Adelaide",
    "TAS1": "Australia/Hobart",
    "WEM": "Australia/Perth",
}

df["ts"] = pd.to_datetime(df["ts"], utc=True)

hour_slot = pd.Series(index=df.index, dtype="float64")
isodow = pd.Series(index=df.index, dtype="float64")  # Monday=1..Sunday=7, matches dbt's isodow
month = pd.Series(index=df.index, dtype="float64")

for region, tz_name in REGION_TZ.items():
    mask = (df["region"] == region).to_numpy()
    local = df.loc[mask, "ts"].dt.tz_convert(ZoneInfo(tz_name))
    hour_slot.loc[mask] = local.dt.hour * 2 + (local.dt.minute // 30)
    isodow.loc[mask] = local.dt.dayofweek + 1
    month.loc[mask] = local.dt.month

df["hour_sin"] = np.sin(np.radians(360.0 * hour_slot / 48))
df["hour_cos"] = np.cos(np.radians(360.0 * hour_slot / 48))
df["dow_sin"] = np.sin(np.radians(360.0 * isodow / 7))
df["dow_cos"] = np.cos(np.radians(360.0 * isodow / 7))
df["month_sin"] = np.sin(np.radians(360.0 * month / 12))
df["month_cos"] = np.cos(np.radians(360.0 * month / 12))

df = df.rename(columns={"ts": "ts_30"})
df[["region", "ts_30", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]].head()


In [ ]:
from ecolens.forecasting.features import FEATURE_COLUMNS

feature_df = df[["region", "ts_30", *FEATURE_COLUMNS]].copy()

assert tuple(feature_df.columns[2:]) == FEATURE_COLUMNS
assert feature_df[list(FEATURE_COLUMNS)].isna().sum().sum() == 0

print(f"feature_df shape: {feature_df.shape}")
feature_df.describe().T


## 3. Window it -- the real `build_windowed_dataset`

Chronological (never shuffled) split within each region first, so every
split has data from every region, then a scaler fit only on the train
split -- exactly `features.py`'s documented contract.

In [ ]:
from ecolens.config import get_settings
from ecolens.forecasting.features import build_windowed_dataset

settings = get_settings()
print(
    f"lookback={settings.model_lookback} horizon={settings.model_horizon} "
    f"hidden_size={settings.model_hidden_size} num_layers={settings.model_num_layers} "
    f"dropout={settings.model_dropout} lr={settings.model_train_lr} "
    f"batch_size={settings.model_batch_size} conformal_alpha={settings.conformal_alpha}"
)

dataset = build_windowed_dataset(
    feature_df, lookback=settings.model_lookback, horizon=settings.model_horizon
)
for split_name in ["train", "val", "calibration", "test"]:
    split = getattr(dataset, split_name)
    print(f"{split_name:11s} x={tuple(split.x.shape)} y={tuple(split.y.shape)}")


## 4. Train the real `DemandLSTM`

`Settings` defaults (`model_train_epochs=50`, `model_early_stop_patience=10`)
are a real production run -- benchmarked at ~50-60s/epoch on this machine
before writing this notebook, so a full run is a background training job
(`training/train.py`, or `training/colab_dispatch.py` for a GPU), not an
interactive notebook cell. Bounded to a handful of epochs here to prove
the pipeline actually converges and produces sane metrics end to end, not
to produce a production-quality checkpoint -- everything else
(architecture, loss, optimizer, MLflow logging) is the unmodified real
training loop.

In [ ]:
from ecolens.forecasting.training.train import train_model

smoke_settings = settings.model_copy(
    update={"model_train_epochs": 8, "model_early_stop_patience": 3}
)

result = train_model(dataset, smoke_settings, log_to_mlflow=True)
print(
    f"run_id={result.run_id} best_val_loss={result.best_val_loss:.4f} "
    f"epochs_trained={result.epochs_trained}"
)


In [ ]:
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(smoke_settings.mlflow_tracking_uri)
client = MlflowClient()
train_hist = client.get_metric_history(result.run_id, "train_loss")
val_hist = client.get_metric_history(result.run_id, "val_loss")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([m.step for m in train_hist], [m.value for m in train_hist], label="train", marker="o")
ax.plot([m.step for m in val_hist], [m.value for m in val_hist], label="val", marker="o")
ax.set_xlabel("epoch")
ax.set_ylabel("loss (Huber P50 + weighted pinball P10/P90)")
ax.set_title("DemandLSTM training curve (bounded smoke run)")
ax.legend()
plt.show()


## 5. Evaluate: conformal calibration + point-forecast metrics

Fits CQR on the calibration split, scores point accuracy and calibrated-
interval coverage on the test split -- the one split that touched
neither training, early-stopping, nor calibration.

In [ ]:
from ecolens.forecasting.evaluation.evaluate import evaluate_model, log_evaluation_to_mlflow

evaluation = evaluate_model(result.model, dataset, alpha=smoke_settings.conformal_alpha)

print("overall:", {k: round(v, 3) for k, v in evaluation.point.overall.items()})
print(f"test coverage: {evaluation.test_coverage:.3f} (target {1 - smoke_settings.conformal_alpha:.2f})")

log_evaluation_to_mlflow(evaluation, run_id=result.run_id, settings=smoke_settings)

print("\nper-region:")
display(evaluation.point.per_region)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(evaluation.point.per_horizon_step["horizon_step"], evaluation.point.per_horizon_step["mae"], label="MAE")
ax.plot(evaluation.point.per_horizon_step["horizon_step"], evaluation.point.per_horizon_step["rmse"], label="RMSE")
ax.set_xlabel("horizon step (30-min ahead)")
ax.set_ylabel("MW")
ax.set_title("Error by forecast horizon -- errors should grow with horizon, not be flat/noisy")
ax.legend()
plt.show()


## 6. A worked example: forecast vs actual, one region, one test window

In [ ]:
from ecolens.forecasting.evaluation.evaluate import predict_split

test_preds = predict_split(result.model, dataset.test, dataset.scaler)

sample_region = "NSW1"
region_positions = np.where((dataset.test.region == sample_region).to_numpy())[0]
idx = region_positions[len(region_positions) // 2]

p10_cal, p90_cal = evaluation.conformal.calibrate(
    test_preds["p10"][idx : idx + 1], test_preds["p90"][idx : idx + 1]
)

horizon_steps = np.arange(1, dataset.horizon + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(horizon_steps, test_preds["y_true"][idx], label="actual", color="black")
ax.plot(horizon_steps, test_preds["p50"][idx], label="p50", color="C0")
ax.fill_between(
    horizon_steps, p10_cal[0], p90_cal[0], alpha=0.2, color="C0",
    label=f"calibrated {int((1 - evaluation.conformal.alpha) * 100)}% interval",
)
ax.set_xlabel("horizon step (30-min ahead)")
ax.set_ylabel("demand_mw")
ax.set_title(f"{sample_region}, test-split window as of {dataset.test.as_of.iloc[idx]}")
ax.legend()
plt.show()


## Summary

What this proves: `ecolens.forecasting`'s `DemandLSTM` + windowing +
training loop + conformal calibration is real, working code that trains,
converges, and produces calibrated forecasts against real historical
demand/weather/holiday data (not scaffolding, not synthetic data).

What's still open, per `TODO.md`:
- **The real `ml_features_demand_v1` mart isn't on NeonDB yet.** This
  notebook's `feature_df` is a local, pandas-side approximation of it --
  once the `raw.*` syncer lands and the mart can actually be built via
  dbt, re-run this same downstream pipeline (`build_windowed_dataset` ->
  `train_model` -> `evaluate_model`) against `TrainingSetLoader`'s output
  instead, and the "not yet against real production data" caveat goes away.
- **This was a bounded smoke run** (8 epochs / patience 3), not a full
  training run against `Settings`' real defaults (50 epochs / patience
  10) -- that belongs in a background job or `training/colab_dispatch.py`,
  not an interactive notebook cell.
- **TFT and TimesFM** ("Stand up the TFT" / "Stand up TimesFM" in
  `TODO.md`) have no code anywhere in this repo yet -- out of scope here,
  this notebook only validates the LSTM.